In [ ]:
# =====================================================================
# PHYSICAL AI SMART FACTORY DEMO WITH GEMINI
# GOOGLE COLAB - SINGLE CELL
#
# Demonstrates:
# - Conveyor belt
# - Moving products
# - Simulated vision inspection
# - Sensor values
# - Gemini LLM decision:
#       ACCEPT
#       REJECT
#       MANUAL_INSPECTION
# - Deterministic safety override
# - Physical reject actuator
# - Closed-loop Physical AI concept
#
# EDUCATIONAL SIMULATION ONLY
# =====================================================================


# =====================================================================
# 1. INSTALL LIBRARIES
# =====================================================================

!pip install -q google-genai opencv-python


# =====================================================================
# 2. IMPORT LIBRARIES
# =====================================================================

import cv2
import numpy as np
import random
import os
import json

from google import genai
from google.colab import userdata

from IPython.display import HTML, display
from base64 import b64encode

from google import genai
from google.colab import userdata

USE_GEMINI = False
client = None


# =====================================================================
# 3. GEMINI SETUP
# =====================================================================

# In Google Colab:
# Left panel -> Secrets
# Add:
#
# GEMINI_API_KEY
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")



In [ ]:

# =====================================================================
# 4. VIDEO SETTINGS
# =====================================================================

WIDTH = 1280
HEIGHT = 720

FPS = 25
DURATION = 20

TOTAL_FRAMES = FPS * DURATION

raw_video = "smart_factory_gemini_raw.mp4"

final_video = "smart_factory_gemini_demo.mp4"


writer = cv2.VideoWriter(

    raw_video,

    cv2.VideoWriter_fourcc(*"mp4v"),

    FPS,

    (WIDTH, HEIGHT)

)


# =====================================================================
# 5. FACTORY STATE
# =====================================================================

conveyor_speed = 8.0

accepted_count = 0
rejected_count = 0
manual_count = 0

machine_temperature = 42.0

machine_vibration = 1.8

inspection_x = 650

reject_x = 930


# =====================================================================
# 6. PRODUCT CLASS
# =====================================================================

class Product:

    def __init__(self, product_id, x):

        self.id = product_id

        self.x = x

        self.y = 420

        self.width = 100

        self.height = 75

        self.product_type = random.choice(
            [
                "Gear",
                "Bearing",
                "Metal-Part"
            ]
        )


        # -------------------------------------------------------------
        # SIMULATED VISION / SENSOR FEATURES
        # -------------------------------------------------------------

        self.crack_score = round(
            random.uniform(0, 1),
            2
        )

        self.surface_score = round(
            random.uniform(0, 1),
            2
        )

        self.dimension_error_mm = round(
            random.uniform(0, 4),
            2
        )

        self.temperature = round(
            random.uniform(35, 70),
            1
        )

        self.inspected = False

        self.gemini_decision = "WAITING"

        self.final_decision = "WAITING"

        self.reason = ""

        self.rejected = False

        self.manual = False

        self.finished = False


# =====================================================================
# 7. INITIAL PRODUCTS
# =====================================================================

products = []

next_product_id = 1


for i in range(5):

    products.append(

        Product(
            next_product_id,
            80 - i * 250
        )

    )

    next_product_id += 1


# =====================================================================
# 8. FALLBACK QUALITY DECISION
# =====================================================================

def fallback_quality_decision(product):

    if (
        product.crack_score > 0.75
        or
        product.dimension_error_mm > 3.0
        or
        product.temperature > 62
    ):

        return (
            "REJECT",
            "Critical defect threshold exceeded"
        )


    elif (
        product.crack_score > 0.45
        or
        product.surface_score > 0.65
        or
        product.dimension_error_mm > 1.8
    ):

        return (
            "MANUAL_INSPECTION",
            "Borderline quality condition"
        )


    else:

        return (
            "ACCEPT",
            "Quality parameters are within acceptable range"
        )


In [ ]:


# =====================================================================
# 9. GEMINI QUALITY DECISION
# =====================================================================

def gemini_quality_decision(product):


    if not USE_GEMINI:

        return fallback_quality_decision(
            product
        )


    product_state = {

        "product_id":
            product.id,

        "product_type":
            product.product_type,

        "crack_score":
            product.crack_score,

        "surface_defect_score":
            product.surface_score,

        "dimension_error_mm":
            product.dimension_error_mm,

        "product_temperature_c":
            product.temperature

    }


    prompt = f"""

You are the HIGH-LEVEL quality reasoning module
inside an EDUCATIONAL Physical AI smart factory.

A computer vision system and factory sensors
have inspected a manufactured product.

Product data:

{json.dumps(product_state, indent=2)}

Choose exactly ONE action:

ACCEPT
REJECT
MANUAL_INSPECTION

Guidelines:

- REJECT if the defect is clearly severe.
- ACCEPT if quality is clearly good.
- MANUAL_INSPECTION if the case is uncertain or borderline.
- Safety and quality are more important than production speed.
- Do not invent additional sensor values.

Return exactly:

ACTION|short reason

Example:

REJECT|Crack score is very high and dimension error exceeds tolerance

"""


    try:

        response = client.models.generate_content(

            model="gemini-2.5-flash",

            contents=prompt

        )


        text = (
            response.text
            or ""
        ).strip()


        if "|" not in text:

            return fallback_quality_decision(
                product
            )


        action, reason = text.split(
            "|",
            1
        )


        action = action.strip().upper()

        reason = reason.strip()


        valid = [

            "ACCEPT",

            "REJECT",

            "MANUAL_INSPECTION"

        ]


        if action not in valid:

            return fallback_quality_decision(
                product
            )


        return action, reason


    except Exception as e:

        print(
            "Gemini decision failed:",
            e
        )

        return fallback_quality_decision(
            product
        )


# =====================================================================
# 10. DETERMINISTIC SAFETY OVERRIDE
# =====================================================================

def safety_override(
    product,
    gemini_decision
):


    # -------------------------------------------------------------
    # Critical crack
    # -------------------------------------------------------------

    if product.crack_score > 0.92:

        return (
            "REJECT",
            "SAFETY OVERRIDE: critical crack score"
        )


    # -------------------------------------------------------------
    # Excessive dimensional error
    # -------------------------------------------------------------

    if product.dimension_error_mm > 3.7:

        return (
            "REJECT",
            "SAFETY OVERRIDE: dimensional tolerance exceeded"
        )


    # -------------------------------------------------------------
    # Excessive temperature
    # -------------------------------------------------------------

    if product.temperature > 68:

        return (
            "REJECT",
            "SAFETY OVERRIDE: product temperature too high"
        )


    return (
        gemini_decision,
        "APPROVED"
    )


# =====================================================================
# 11. FACTORY MACHINE CONTROLLER
# =====================================================================

def factory_controller():

    global machine_temperature
    global machine_vibration
    global conveyor_speed


    machine_temperature += random.uniform(
        -0.08,
        0.10
    )


    machine_vibration += random.uniform(
        -0.03,
        0.03
    )


    if machine_temperature > 60:

        conveyor_speed = 3

        return (
            "SLOW LINE - MACHINE TEMPERATURE HIGH"
        )


    if machine_vibration > 4:

        conveyor_speed = 3

        return (
            "SLOW LINE - VIBRATION HIGH"
        )


    conveyor_speed = 8

    return "NORMAL"


In [ ]:


# =====================================================================
# 12. DRAW FACTORY
# =====================================================================

def draw_factory(frame):


    frame[:] = (
        225,
        225,
        225
    )


    cv2.putText(

        frame,

        "PHYSICAL AI - SMART FACTORY WITH GEMINI",

        (35,55),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.9,

        (20,20,20),

        3

    )


    # Conveyor

    cv2.rectangle(

        frame,

        (40,450),

        (1190,560),

        (70,70,70),

        -1

    )


    cv2.rectangle(

        frame,

        (40,455),

        (1190,540),

        (110,110,110),

        -1

    )


    for x in range(
        60,
        1180,
        70
    ):

        cv2.circle(

            frame,

            (x,555),

            18,

            (40,40,40),

            -1

        )


    # -------------------------------------------------------------
    # AI camera station
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (
            inspection_x-70,
            175
        ),

        (
            inspection_x+70,
            265
        ),

        (60,60,60),

        -1

    )


    cv2.circle(

        frame,

        (
            inspection_x,
            220
        ),

        28,

        (0,100,255),

        -1

    )


    cv2.circle(

        frame,

        (
            inspection_x,
            220
        ),

        15,

        (0,0,0),

        -1

    )


    cv2.line(

        frame,

        (
            inspection_x,
            265
        ),

        (
            inspection_x,
            420
        ),

        (0,255,255),

        3

    )


    cv2.putText(

        frame,

        "VISION + SENSOR STATION",

        (
            inspection_x-125,
            150
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.6,

        (0,0,0),

        2

    )


    # -------------------------------------------------------------
    # Reject actuator
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (
            reject_x-30,
            320
        ),

        (
            reject_x+30,
            410
        ),

        (120,120,120),

        -1

    )


    cv2.putText(

        frame,

        "ACTUATOR",

        (
            reject_x-55,
            295
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.55,

        (0,0,0),

        2

    )


    # -------------------------------------------------------------
    # Reject bin
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (1010,570),

        (1180,690),

        (80,80,180),

        -1

    )


    cv2.putText(

        frame,

        "REJECT",

        (1050,625),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.7,

        (255,255,255),

        2

    )


    # -------------------------------------------------------------
    # Manual inspection bin
    # -------------------------------------------------------------

    cv2.rectangle(

        frame,

        (800,570),

        (970,690),

        (80,160,180),

        -1

    )


    cv2.putText(

        frame,

        "MANUAL",

        (825,620),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (255,255,255),

        2

    )


    cv2.putText(

        frame,

        "INSPECTION",

        (810,650),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255,255,255),

        1

    )


# =====================================================================
# 13. DRAW PRODUCT
# =====================================================================

def draw_product(
    frame,
    product
):


    x = int(
        product.x
    )

    y = int(
        product.y
    )


    if not product.inspected:

        color = (
            180,
            140,
            60
        )


    elif product.final_decision == "ACCEPT":

        color = (
            0,
            180,
            0
        )


    elif product.final_decision == "REJECT":

        color = (
            0,
            0,
            220
        )


    else:

        color = (
            0,
            180,
            220
        )


    cv2.rectangle(

        frame,

        (x,y),

        (
            x+product.width,
            y+product.height
        ),

        color,

        -1

    )


    # Bounding box

    cv2.rectangle(

        frame,

        (
            x-5,
            y-5
        ),

        (
            x+product.width+5,
            y+product.height+5
        ),

        (0,255,255),

        2

    )


    label = (

        f"{product.product_type} "
        f"ID:{product.id}"

    )


    cv2.putText(

        frame,

        label,

        (
            x,
            y-12
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.45,

        (0,0,0),

        2

    )


    if product.inspected:

        cv2.putText(

            frame,

            product.final_decision,

            (
                x+5,
                y+42
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.48,

            (255,255,255),

            2

        )


# =====================================================================
# 14. DASHBOARD
# =====================================================================

def draw_dashboard(

    frame,
    factory_status,
    active_product,
    safety_message

):


    cv2.rectangle(

        frame,

        (25,75),

        (575,365),

        (25,25,25),

        -1

    )


    cv2.putText(

        frame,

        "GENAI + PHYSICAL AI CONTROL",

        (45,110),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.68,

        (0,255,0),

        2

    )


    rows = [

        f"Conveyor speed : {conveyor_speed:.1f} m/min",

        f"Machine temp   : {machine_temperature:.1f} C",

        f"Vibration      : {machine_vibration:.2f}",

        f"Accepted       : {accepted_count}",

        f"Rejected       : {rejected_count}",

        f"Manual inspect : {manual_count}",

        f"Machine status : {factory_status}"

    ]


    y = 150


    for row in rows:

        cv2.putText(

            frame,

            row,

            (45,y),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.52,

            (255,255,255),

            1

        )

        y += 30


    # -------------------------------------------------------------
    # Gemini decision panel
    # -------------------------------------------------------------

    if active_product is not None:


        cv2.rectangle(

            frame,

            (650,70),

            (1240,310),

            (245,245,245),

            -1

        )


        cv2.putText(

            frame,

            "GEMINI QUALITY REASONING",

            (680,105),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.66,

            (0,0,0),

            2

        )


        rows = [

            f"Product: {active_product.product_type}",

            f"Crack score: {active_product.crack_score:.2f}",

            f"Surface defect: {active_product.surface_score:.2f}",

            f"Dimension error: {active_product.dimension_error_mm:.2f} mm",

            f"Temperature: {active_product.temperature:.1f} C",

            f"Gemini: {active_product.gemini_decision}",

            f"Final: {active_product.final_decision}",

            f"Safety: {safety_message}"

        ]


        y = 140


        for row in rows:

            cv2.putText(

                frame,

                row[:80],

                (680,y),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.48,

                (20,20,20),

                1

            )

            y += 24


# =====================================================================
# 15. MAIN SIMULATION
# =====================================================================

active_product = None

active_safety_message = "APPROVED"


for frame_number in range(
    TOTAL_FRAMES
):


    frame = np.zeros(

        (
            HEIGHT,
            WIDTH,
            3
        ),

        dtype=np.uint8

    )


    draw_factory(
        frame
    )


    factory_status = factory_controller()


    for product in products:


        if product.finished:

            continue


        # -------------------------------------------------------------
        # Conveyor movement
        # -------------------------------------------------------------

        product.x += (
            conveyor_speed
            *
            0.55
        )


        # =============================================================
        # SENSE -> REASON
        # =============================================================

        if (

            not product.inspected

            and

            product.x
            >=
            inspection_x-45

        ):


            # ---------------------------------------------------------
            # GEMINI DECISION
            # ---------------------------------------------------------

            gemini_decision, reason = (
                gemini_quality_decision(
                    product
                )
            )


            product.gemini_decision = (
                gemini_decision
            )


            product.reason = reason


            # ---------------------------------------------------------
            # SAFETY CHECK
            # ---------------------------------------------------------

            final_decision, safety_message = (

                safety_override(

                    product,

                    gemini_decision

                )

            )


            product.final_decision = (
                final_decision
            )


            product.inspected = True


            active_product = product

            active_safety_message = (
                safety_message
            )


            print()

            print(
                "="*70
            )

            print(
                f"Product {product.id}"
            )

            print(
                "Type:",
                product.product_type
            )

            print(
                "Crack:",
                product.crack_score
            )

            print(
                "Surface:",
                product.surface_score
            )

            print(
                "Dimension error:",
                product.dimension_error_mm
            )

            print(
                "Temperature:",
                product.temperature
            )

            print(
                "Gemini:",
                gemini_decision
            )

            print(
                "Reason:",
                reason
            )

            print(
                "Final:",
                final_decision
            )

            print(
                "Safety:",
                safety_message
            )


        # =============================================================
        # ACT
        # =============================================================

        if (

            product.inspected

            and

            not product.rejected

            and

            not product.manual

            and

            product.x >= reject_x

        ):


            # ---------------------------------------------------------
            # REJECT
            # ---------------------------------------------------------

            if (
                product.final_decision
                ==
                "REJECT"
            ):


                product.rejected = True

                product.y += 150

                rejected_count += 1


            # ---------------------------------------------------------
            # MANUAL INSPECTION
            # ---------------------------------------------------------

            elif (
                product.final_decision
                ==
                "MANUAL_INSPECTION"
            ):


                product.manual = True

                product.y += 150

                manual_count += 1


        # -------------------------------------------------------------
        # Accepted product leaves line
        # -------------------------------------------------------------

        if (

            product.final_decision
            ==
            "ACCEPT"

            and

            product.x > WIDTH

            and

            not product.finished

        ):


            accepted_count += 1

            product.finished = True


        # -------------------------------------------------------------
        # Rejected / manual product finishes
        # -------------------------------------------------------------

        if (

            (
                product.rejected
                or
                product.manual
            )

            and

            product.x > 1120

        ):


            product.finished = True


        draw_product(

            frame,

            product

        )


    # =================================================================
    # ADD NEW PRODUCTS
    # =================================================================

    if (

        frame_number % 90 == 0

        and

        frame_number > 0

    ):


        products.append(

            Product(

                next_product_id,

                -130

            )

        )


        next_product_id += 1


    # =================================================================
    # DRAW ACTUATOR
    # =================================================================

    actuator_active = any(

        (
            p.rejected
            or
            p.manual
        )

        and

        reject_x-60
        <
        p.x
        <
        reject_x+140

        for p in products

    )


    if actuator_active:


        cv2.arrowedLine(

            frame,

            (
                reject_x,
                390
            ),

            (
                reject_x+80,
                490
            ),

            (0,0,255),

            10,

            tipLength=0.25

        )


        cv2.putText(

            frame,

            "PHYSICAL ACTUATOR ACTIVE",

            (
                reject_x-140,
                375
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.50,

            (0,0,255),

            2

        )


    # =================================================================
    # DASHBOARD
    # =================================================================

    draw_dashboard(

        frame,

        factory_status,

        active_product,

        active_safety_message

    )


    cv2.putText(

        frame,

        "Sense -> Perceive -> Gemini Reason -> Safety -> Act -> Feedback",

        (
            300,
            695
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.68,

        (20,20,20),

        2

    )


    writer.write(
        frame
    )


# =====================================================================
# 16. CLOSE VIDEO
# =====================================================================

writer.release()


# =====================================================================
# 17. CONVERT TO H264
# =====================================================================

os.system(

    f'ffmpeg -y '
    f'-loglevel error '
    f'-i "{raw_video}" '
    f'-vcodec libx264 '
    f'-pix_fmt yuv420p '
    f'"{final_video}"'

)


# =====================================================================
# 18. DISPLAY VIDEO
# =====================================================================

with open(
    final_video,
    "rb"
) as f:

    video_bytes = f.read()


encoded_video = b64encode(
    video_bytes
).decode()


print()
print(
    "SMART FACTORY GEMINI PHYSICAL AI DEMO"
)


display(

    HTML(

        f'''

        <video
            width="1000"
            controls
            autoplay
            loop>

            <source
                src="data:video/mp4;base64,{encoded_video}"
                type="video/mp4">

        </video>

        '''

    )

)

In [ ]:

# =====================================================================
# VIDEO SETTINGS
# =====================================================================

WIDTH = 1280
HEIGHT = 720
FPS = 25
DURATION = 20

TOTAL_FRAMES = FPS * DURATION

raw_video = "smart_factory_raw.mp4"
final_video = "smart_factory_gemini_demo.mp4"

writer = cv2.VideoWriter(
    raw_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (WIDTH, HEIGHT)
)


# =====================================================================
# FACTORY STATE
# =====================================================================

conveyor_speed = 8.0

accepted_count = 0
rejected_count = 0
manual_count = 0

machine_temperature = 42.0
machine_vibration = 1.8

inspection_x = 650
reject_x = 930


# =====================================================================
# PRODUCT
# =====================================================================

class Product:

    def __init__(self, product_id, x):

        self.id = product_id

        self.x = x
        self.y = 420

        self.width = 100
        self.height = 65

        self.product_type = random.choice(
            ["Gear", "Bearing", "Metal-Part"]
        )

        self.crack_score = round(
            random.uniform(0,1),2
        )

        self.surface_score = round(
            random.uniform(0,1),2
        )

        self.dimension_error_mm = round(
            random.uniform(0,4),2
        )

        self.temperature = round(
            random.uniform(35,70),1
        )

        self.inspected = False

        self.gemini_decision = "WAITING"
        self.final_decision = "WAITING"

        self.reason = ""

        self.rejected = False
        self.manual = False
        self.finished = False


# =====================================================================
# INITIAL PRODUCTS
# =====================================================================

products = []

next_product_id = 1

for i in range(5):

    products.append(
        Product(
            next_product_id,
            80 - i*250
        )
    )

    next_product_id += 1


# =====================================================================
# FALLBACK QUALITY DECISION
# =====================================================================

def fallback_quality_decision(product):

    if (
        product.crack_score > 0.75
        or product.dimension_error_mm > 3.0
        or product.temperature > 62
    ):

        return (
            "REJECT",
            "Critical defect threshold exceeded"
        )

    elif (
        product.crack_score > 0.45
        or product.surface_score > 0.65
        or product.dimension_error_mm > 1.8
    ):

        return (
            "MANUAL_INSPECTION",
            "Borderline quality condition"
        )

    return (
        "ACCEPT",
        "Quality parameters acceptable"
    )


# =====================================================================
# GEMINI QUALITY REASONING
# =====================================================================

def gemini_quality_decision(product):

    if not USE_GEMINI:
        return fallback_quality_decision(product)

    product_state = {

        "product_type":
            product.product_type,

        "crack_score":
            product.crack_score,

        "surface_defect_score":
            product.surface_score,

        "dimension_error_mm":
            product.dimension_error_mm,

        "temperature_c":
            product.temperature
    }

    prompt = f"""
You are the quality reasoning module of an
educational Physical AI smart factory.

Sensor data:

{json.dumps(product_state, indent=2)}

Choose exactly one:

ACCEPT
REJECT
MANUAL_INSPECTION

Guidelines:

REJECT severe defects.

ACCEPT clearly good products.

MANUAL_INSPECTION when uncertain.

Return exactly:

ACTION|short reason
"""

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        text = (response.text or "").strip()

        if "|" not in text:
            return fallback_quality_decision(product)

        action, reason = text.split("|",1)

        action = action.strip().upper()
        reason = reason.strip()

        if action not in [
            "ACCEPT",
            "REJECT",
            "MANUAL_INSPECTION"
        ]:

            return fallback_quality_decision(product)

        return action, reason

    except Exception:

        return fallback_quality_decision(product)


# =====================================================================
# SAFETY OVERRIDE
# =====================================================================

def safety_override(product, decision):

    if product.crack_score > 0.92:

        return (
            "REJECT",
            "OVERRIDE: critical crack"
        )

    if product.dimension_error_mm > 3.7:

        return (
            "REJECT",
            "OVERRIDE: tolerance exceeded"
        )

    if product.temperature > 68:

        return (
            "REJECT",
            "OVERRIDE: temperature high"
        )

    return (
        decision,
        "APPROVED"
    )


# =====================================================================
# MACHINE CONTROLLER
# =====================================================================

def factory_controller():

    global machine_temperature
    global machine_vibration
    global conveyor_speed

    machine_temperature += random.uniform(
        -0.08,
        0.10
    )

    machine_vibration += random.uniform(
        -0.03,
        0.03
    )

    if machine_temperature > 60:

        conveyor_speed = 3

        return "SLOW LINE - HIGH TEMP"

    if machine_vibration > 4:

        conveyor_speed = 3

        return "SLOW LINE - HIGH VIBRATION"

    conveyor_speed = 8

    return "NORMAL"


# =====================================================================
# FACTORY DRAWING
# =====================================================================

def draw_factory(frame):

    frame[:] = (238,238,238)

    cv2.rectangle(
        frame,
        (0,560),
        (WIDTH,HEIGHT),
        (220,220,220),
        -1
    )

    cv2.putText(
        frame,
        "PHYSICAL AI - SMART FACTORY WITH GEMINI",
        (75,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.95,
        (15,15,15),
        3,
        cv2.LINE_AA
    )


    # ================================================================
    # VISION SENSOR
    # ================================================================

    cv2.putText(
        frame,
        "VISION + SENSOR STATION",
        (500,115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.64,
        (20,20,20),
        2
    )

    camera_x = inspection_x
    camera_y = 175

    # camera
    cv2.rectangle(
        frame,
        (camera_x-58,camera_y),
        (camera_x+58,camera_y+95),
        (45,45,45),
        -1
    )

    cv2.rectangle(
        frame,
        (camera_x-45,camera_y+10),
        (camera_x+45,camera_y+82),
        (70,70,70),
        -1
    )

    cv2.circle(
        frame,
        (camera_x,camera_y+48),
        30,
        (100,100,100),
        -1
    )

    cv2.circle(
        frame,
        (camera_x,camera_y+48),
        23,
        (190,90,20),
        -1
    )

    cv2.circle(
        frame,
        (camera_x,camera_y+48),
        12,
        (10,10,10),
        -1
    )

    # camera stand
    cv2.rectangle(
        frame,
        (camera_x-14,camera_y+95),
        (camera_x+14,camera_y+120),
        (55,55,55),
        -1
    )


    # ================================================================
    # SENSOR BEAM
    # ================================================================

    overlay = frame.copy()

    beam = np.array(
        [
            [camera_x-12,camera_y+120],
            [camera_x+12,camera_y+120],
            [camera_x+75,445],
            [camera_x-75,445]
        ],
        np.int32
    )

    cv2.fillPoly(
        overlay,
        [beam],
        (255,255,80)
    )

    cv2.addWeighted(
        overlay,
        0.17,
        frame,
        0.83,
        0,
        frame
    )


    # ================================================================
    # CONVEYOR
    # ================================================================

    cv2.rectangle(
        frame,
        (20,445),
        (1260,465),
        (55,55,55),
        -1
    )

    cv2.rectangle(
        frame,
        (20,465),
        (1260,545),
        (40,40,40),
        -1
    )

    cv2.rectangle(
        frame,
        (20,545),
        (1260,590),
        (55,55,55),
        -1
    )


    # rollers
    for x in range(45,1240,70):

        cv2.circle(
            frame,
            (x,566),
            15,
            (20,20,20),
            -1
        )

        cv2.circle(
            frame,
            (x,566),
            9,
            (130,130,130),
            -1
        )


    # ================================================================
    # ACTUATOR
    # ================================================================

    actuator_x = reject_x

    cv2.putText(
        frame,
        "ACTUATOR",
        (actuator_x-55,300),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.58,
        (25,25,25),
        2
    )

    cv2.rectangle(
        frame,
        (actuator_x-24,320),
        (actuator_x+24,370),
        (85,85,85),
        -1
    )

    cv2.rectangle(
        frame,
        (actuator_x-9,370),
        (actuator_x+9,445),
        (100,100,100),
        -1
    )

    cv2.line(
        frame,
        (actuator_x+18,365),
        (actuator_x+85,450),
        (45,45,45),
        15
    )


    # ================================================================
    # MANUAL INSPECTION BIN
    # ================================================================

    cv2.rectangle(
        frame,
        (800,595),
        (965,700),
        (120,110,30),
        -1
    )

    cv2.rectangle(
        frame,
        (790,585),
        (975,605),
        (70,70,40),
        -1
    )

    cv2.putText(
        frame,
        "MANUAL",
        (825,645),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.58,
        (255,255,255),
        2
    )

    cv2.putText(
        frame,
        "INSPECTION",
        (810,675),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (255,255,255),
        2
    )


    # ================================================================
    # REJECT BIN
    # ================================================================

    cv2.rectangle(
        frame,
        (1030,595),
        (1200,700),
        (130,60,30),
        -1
    )

    cv2.rectangle(
        frame,
        (1020,585),
        (1210,605),
        (65,45,30),
        -1
    )

    cv2.putText(
        frame,
        "REJECT",
        (1065,660),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.68,
        (255,255,255),
        2
    )


# =====================================================================
# PRODUCT DRAWING
# =====================================================================

def draw_product(frame, product):

    x = int(product.x)
    y = int(product.y)

    if not product.inspected:

        color = (40,170,210)

    elif product.final_decision == "ACCEPT":

        color = (40,180,80)

    elif product.final_decision == "REJECT":

        color = (50,70,210)

    else:

        color = (40,180,200)


    w = product.width
    h = product.height


    # component body
    cv2.rectangle(
        frame,
        (x,y),
        (x+w,y+h),
        color,
        -1
    )


    # top ribs
    for rib_x in range(
        x+8,
        x+w-5,
        13
    ):

        cv2.rectangle(
            frame,
            (rib_x,y-10),
            (rib_x+7,y+5),
            (
                max(color[0]-30,0),
                max(color[1]-30,0),
                max(color[2]-30,0)
            ),
            -1
        )


    # side details
    cv2.circle(
        frame,
        (x+12,y+h//2),
        6,
        (40,40,40),
        2
    )

    cv2.circle(
        frame,
        (x+w-12,y+h//2),
        6,
        (40,40,40),
        2
    )


    # CV bounding box
    cv2.rectangle(
        frame,
        (x-5,y-15),
        (x+w+5,y+h+5),
        (0,255,255),
        2
    )


    cv2.putText(
        frame,
        f"{product.product_type} ID:{product.id}",
        (x,y-20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.42,
        (20,20,20),
        1
    )


    if product.inspected:

        cv2.putText(
            frame,
            product.final_decision,
            (x+8,y+42),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.40,
            (255,255,255),
            2
        )


# =====================================================================
# SMALLER DASHBOARD PANELS
# =====================================================================

def draw_dashboard(
    frame,
    factory_status,
    active_product,
    safety_message
):

    # ================================================================
    # LEFT CONTROL PANEL
    # ================================================================

    left_x1 = 25
    left_y1 = 75

    left_x2 = 410
    left_y2 = 360


    # shadow
    cv2.rectangle(
        frame,
        (31,81),
        (416,366),
        (120,120,120),
        -1
    )

    # panel
    cv2.rectangle(
        frame,
        (left_x1,left_y1),
        (left_x2,left_y2),
        (28,28,28),
        -1
    )

    cv2.rectangle(
        frame,
        (left_x1,left_y1),
        (left_x2,left_y2),
        (75,75,75),
        3
    )


    cv2.putText(
        frame,
        "GENAI + PHYSICAL AI",
        (45,110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (80,255,80),
        2
    )

    cv2.putText(
        frame,
        "CONTROL",
        (45,135),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (80,255,80),
        2
    )


    cv2.line(
        frame,
        (45,145),
        (390,145),
        (100,100,100),
        1
    )


    rows = [

        f"Conveyor : {conveyor_speed:.1f} m/min",

        f"Temp     : {machine_temperature:.1f} C",

        f"Vibration: {machine_vibration:.2f}",

        f"Accepted : {accepted_count}",

        f"Rejected : {rejected_count}",

        f"Manual   : {manual_count}",

        f"Status   : {factory_status}"
    ]


    y = 175

    for row in rows:

        cv2.putText(
            frame,
            row[:45],
            (45,y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.46,
            (245,245,245),
            1
        )

        y += 27


    # ================================================================
    # RIGHT GEMINI PANEL
    # ================================================================

    if active_product is not None:

        right_x1 = 870
        right_y1 = 75

        right_x2 = 1245
        right_y2 = 345


        # shadow
        cv2.rectangle(
            frame,
            (876,81),
            (1251,351),
            (155,155,155),
            -1
        )


        cv2.rectangle(
            frame,
            (right_x1,right_y1),
            (right_x2,right_y2),
            (247,247,247),
            -1
        )


        cv2.rectangle(
            frame,
            (right_x1,right_y1),
            (right_x2,right_y2),
            (45,45,45),
            3
        )


        cv2.putText(
            frame,
            "GEMINI QUALITY REASONING",
            (890,110),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (170,60,20),
            2
        )


        rows = [

            f"Product       : {active_product.product_type}",

            f"Crack score   : {active_product.crack_score:.2f}",

            f"Surface score : {active_product.surface_score:.2f}",

            f"Dim. error    : {active_product.dimension_error_mm:.2f} mm",

            f"Temperature   : {active_product.temperature:.1f} C",

            f"Gemini        : {active_product.gemini_decision}",

            f"Final         : {active_product.final_decision}",

            f"Safety        : {safety_message}"
        ]


        y = 145

        for row in rows:

            cv2.putText(
                frame,
                row[:52],
                (890,y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.43,
                (25,25,25),
                1
            )

            y += 25


# =====================================================================
# MAIN SIMULATION
# =====================================================================

active_product = None
active_safety_message = "APPROVED"


for frame_number in range(
    TOTAL_FRAMES
):

    frame = np.zeros(
        (HEIGHT,WIDTH,3),
        dtype=np.uint8
    )


    draw_factory(frame)

    factory_status = factory_controller()


    for product in products:


        if product.finished:
            continue


        # conveyor movement
        product.x += (
            conveyor_speed * 0.55
        )


        # ============================================================
        # INSPECTION
        # ============================================================

        if (
            not product.inspected
            and
            product.x >= inspection_x-45
        ):

            decision, reason = (
                gemini_quality_decision(
                    product
                )
            )

            product.gemini_decision = decision

            product.reason = reason


            final_decision, safety_message = (
                safety_override(
                    product,
                    decision
                )
            )

            product.final_decision = (
                final_decision
            )

            product.inspected = True

            active_product = product

            active_safety_message = (
                safety_message
            )


            print(
                f"Product {product.id} | "
                f"{product.product_type} | "
                f"Gemini={decision} | "
                f"Final={final_decision} | "
                f"{reason}"
            )


        # ============================================================
        # ACTUATOR DECISION
        # ============================================================

        if (
            product.inspected
            and
            not product.rejected
            and
            not product.manual
            and
            product.x >= reject_x
        ):

            if product.final_decision == "REJECT":

                product.rejected = True

                product.y += 150

                rejected_count += 1


            elif product.final_decision == "MANUAL_INSPECTION":

                product.manual = True

                product.y += 150

                manual_count += 1


        # ============================================================
        # ACCEPTED PRODUCT
        # ============================================================

        if (
            product.final_decision == "ACCEPT"
            and
            product.x > WIDTH
            and
            not product.finished
        ):

            accepted_count += 1

            product.finished = True


        # ============================================================
        # FINISHED REJECT / MANUAL
        # ============================================================

        if (
            (product.rejected or product.manual)
            and
            product.x > 1120
        ):

            product.finished = True


        draw_product(
            frame,
            product
        )


    # =================================================================
    # ADD NEW PRODUCTS
    # =================================================================

    if (
        frame_number % 90 == 0
        and
        frame_number > 0
    ):

        products.append(
            Product(
                next_product_id,
                -130
            )
        )

        next_product_id += 1


    # =================================================================
    # ACTUATOR ACTIVE VISUAL
    # =================================================================

    actuator_active = any(

        (
            p.rejected
            or
            p.manual
        )

        and

        reject_x-60
        <
        p.x
        <
        reject_x+140

        for p in products
    )


    if actuator_active:

        cv2.arrowedLine(
            frame,
            (reject_x,390),
            (reject_x+90,490),
            (0,0,255),
            9,
            tipLength=0.25
        )

        cv2.putText(
            frame,
            "ACTUATOR ACTIVE",
            (reject_x-115,380),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (0,0,255),
            2
        )


    # =================================================================
    # DASHBOARD
    # =================================================================

    draw_dashboard(
        frame,
        factory_status,
        active_product,
        active_safety_message
    )


    # =================================================================
    # PHYSICAL AI LOOP
    # =================================================================

    cv2.putText(
        frame,
        "Sense -> Perceive -> Gemini Reason -> Safety -> Act -> Feedback",
        (300,710),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.62,
        (20,20,20),
        2
    )


    writer.write(frame)


# =====================================================================
# FINISH VIDEO
# =====================================================================

writer.release()


# =====================================================================
# CONVERT TO H264
# =====================================================================

os.system(

    f'ffmpeg -y '
    f'-loglevel error '
    f'-i "{raw_video}" '
    f'-vcodec libx264 '
    f'-pix_fmt yuv420p '
    f'"{final_video}"'

)


# =====================================================================
# DISPLAY VIDEO
# =====================================================================

with open(
    final_video,
    "rb"
) as f:

    video_bytes = f.read()


encoded_video = b64encode(
    video_bytes
).decode()


print()
print(
    "SMART FACTORY PHYSICAL AI + GEMINI DEMO"
)


display(

    HTML(

        f'''
        <video
            width="1000"
            controls
            autoplay
            loop>

            <source
                src="data:video/mp4;base64,{encoded_video}"
                type="video/mp4">

        </video>
        '''

    )

)